# F1-SIS Safety Car Risk Estimation

This notebook estimates calibrated probabilities of a **new** Safety Car within 1, 3, 5, and 10 laps. It is a scenario-weighting input for Monte Carlo strategy simulation, not a crash predictor.

The central experiment is: does causal current-race evidence improve a historical SC-risk prior?

## Modelling approach

For every forecast offset $k$, a historical prior is estimated from circuit and causal race progress. Current evidence updates—but cannot replace—that prior:

$$\operatorname{logit}(h(t+k)) = \operatorname{logit}(p_{historical}(t+k)) + \beta X_t.$$

The coefficient on the historical-prior logit is fixed at one. This is an extension of historical race-progress SC modelling (including the Heilmeier-inspired approach), not a black-box replacement.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import expit, logit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss, log_loss, average_precision_score, roc_auc_score
from sklearn.calibration import calibration_curve
warnings.filterwarnings('ignore')
DATA_DIR = Path('data_fastf1_v1/laps'); HORIZONS=(1,3,5,10); MAX_H=max(HORIZONS); DEFAULT_LAPS=60
assert DATA_DIR.exists(); sns.set_theme(style='whitegrid')

## Temporary raw-data adapter

This converts driver-lap CSVs to causal race-level states. It is deliberately a replaceable adapter: the future Race State Manager should provide the same state fields directly.

In [ ]:
def load_laps(data_dir=DATA_DIR):
    out=[]
    for path in sorted(data_dir.glob('*/*.csv')):
        x=pd.read_csv(path, low_memory=False)
        if len(x) and 'LapNumber' in x:
            x['race_id']=f"{int(x.Year.iloc[0])}_{int(x.Round.iloc[0]):02d}"; x['race_name']=x.GrandPrix.iloc[0]; out.append(x)
    return pd.concat(out, ignore_index=True)

def status_text(x): return str(int(float(x))) if pd.notna(x) else ''

def build_states(raw):
    x=raw[raw.LapNumber.notna() & raw.LapNumber.gt(0)].copy(); x['lap']=x.LapNumber.astype(int)
    x['status']=x.TrackStatus.map(status_text); x['sc_driver']=x.status.str.contains('4', regex=False); x['vsc_driver']=x.status.str.contains('6', regex=False); x['yellow_driver']=x.status.str.contains('2', regex=False)
    s=x.groupby(['race_id','lap'],as_index=False).agg(year=('Year','first'),round=('Round','first'),circuit=('Location','first'),race_name=('race_name','first'),active_cars=('Driver','nunique'),sc_active=('sc_driver','max'),vsc_active=('vsc_driver','max'),yellow_active=('yellow_driver','max'),median_lap_time=('LapTimeSeconds','median'),lap_time_std=('LapTimeSeconds','std'),median_sector_1=('Sector1TimeSeconds','median'),median_sector_2=('Sector2TimeSeconds','median'),median_sector_3=('Sector3TimeSeconds','median'),median_gap_ahead=('IntervalToPositionAheadSeconds','median'),close_battle_rate=('IntervalToPositionAheadSeconds',lambda z:z.le(1.0).mean()),pit_stops_lap=('PitInTimeSeconds',lambda z:z.notna().sum()),median_tyre_life=('TyreLife','median'),rainfall=('Rainfall','max'),track_temp=('TrackTemp','median'),air_temp=('AirTemp','median'))
    s=s.sort_values(['race_id','lap']).reset_index(drop=True); g=s.groupby('race_id',group_keys=False)
    s['observed_final_lap']=g.lap.transform('max') # diagnostic only; never a feature
    s['sc_start']=s.sc_active & ~g.sc_active.shift(fill_value=False); s['vsc_start']=s.vsc_active & ~g.vsc_active.shift(fill_value=False)
    s['previous_sc_count']=g.sc_start.cumsum()-s.sc_start.astype(int); s['previous_vsc_count']=g.vsc_start.cumsum()-s.vsc_start.astype(int)
    s['laps_since_sc']=g.sc_start.transform(lambda z:(~z).groupby(z.cumsum()).cumcount()+1); s['laps_since_vsc']=g.vsc_start.transform(lambda z:(~z).groupby(z.cumsum()).cumcount()+1)
    s['recent_pits_3']=g.pit_stops_lap.transform(lambda z:z.rolling(3,min_periods=1).sum()); s['pace_abnormality']=g.median_lap_time.transform(lambda z:z/z.rolling(5,min_periods=2).median()-1); s['weather_change']=g.track_temp.diff().fillna(0)
    return s

def attach_causal_lap_plan(s):
    races=s[['race_id','year','round','circuit','observed_final_lap']].drop_duplicates().sort_values(['year','round']); prior_all=[]; prior_circuit={}; plan={}
    for r in races.itertuples(index=False):
        history=prior_circuit.get(r.circuit,[]); plan[r.race_id]=int(round(np.median(history if history else prior_all))) if (history or prior_all) else DEFAULT_LAPS
        prior_circuit.setdefault(r.circuit,[]).append(r.observed_final_lap); prior_all.append(r.observed_final_lap)
    s=s.copy(); s['planned_race_laps']=s.race_id.map(plan); s['race_progress']=(s.lap/s.planned_race_laps).clip(upper=1); return s

raw=load_laps(); states=attach_causal_lap_plan(build_states(raw)); print(states.race_id.nunique(), 'races;',len(states),'race-laps;',int(states.sc_start.sum()),'SC starts')

## TrackStatus validation and targets

`TrackStatus 4` is the authoritative SC-active ground truth; `SC_START` is its inactive-to-active transition. VSC (`6`) is detected separately. Right-censored horizon targets are left missing, never labelled negative.

In [ ]:
def add_targets(s):
    s=s.copy().sort_values(['race_id','lap'])
    for h in HORIZONS:
        s[f'target_{h}']=s.groupby('race_id').sc_start.transform(lambda z:z.iloc[::-1].rolling(h,min_periods=h).max().iloc[::-1].shift(-1))
    return s

states=add_targets(states)
known=states.query("year==2021 and race_name.isin(['Abu Dhabi Grand Prix','Italian Grand Prix']) and (sc_start or vsc_start)")
display(known[['race_name','lap','sc_active','sc_start','vsc_active','vsc_start']])
def split_races(s):
    ids=s[['race_id','year','round']].drop_duplicates().sort_values(['year','round']).race_id.tolist(); a,b=int(.6*len(ids)),int(.8*len(ids)); return ids[:a],ids[a:b],ids[b:]
train_ids,val_ids,test_ids=split_races(states); train=states[states.race_id.isin(train_ids)].copy(); val=states[states.race_id.isin(val_ids)].copy(); test=states[states.race_id.isin(test_ids)].copy()
print('train/validation/test races:',len(train_ids),len(val_ids),len(test_ids),'next-lap prevalence:',train.target_1.mean())

## Historical SC prior

The prior is estimated on training races only, with shrinkage to the global rate. It is evaluated at the deterministic planned progress of each forecast offset. Circuit-prior gaps fall back to race-progress risk.

In [ ]:
def fit_prior(s, circuit=False, bins=12, smoothing=10):
    x=s.dropna(subset=['target_1']).copy(); x['bin']=pd.cut(x.race_progress,np.linspace(0,1,bins+1),include_lowest=True); keys=['bin']+(['circuit'] if circuit else []); global_rate=x.target_1.mean(); t=x.groupby(keys,observed=True).target_1.agg(['sum','count']).reset_index(); t['p']=(t['sum']+smoothing*global_rate)/(t['count']+smoothing); return t[keys+['p']],keys,global_rate
def prior_at(s, fitted, offset):
    table,keys,fallback=fitted; x=s[['circuit','lap','planned_race_laps']].copy(); x['bin']=pd.cut(((x.lap+offset)/x.planned_race_laps).clip(upper=1),np.linspace(0,1,13),include_lowest=True); return x.merge(table,on=keys,how='left').p.fillna(fallback).to_numpy()
progress_prior=fit_prior(train); circuit_prior=fit_prior(train,circuit=True)
fig,ax=plt.subplots(figsize=(8,3)); t=progress_prior[0].copy(); ax.plot([z.mid for z in t.bin],t.p,marker='o'); ax.set(title='Training historical SC-start prior by causal race progress',xlabel='Race progress',ylabel='Next-lap SC-start probability'); plt.show()

## Historical prior + live evidence update

Only plausible causal evidence is included: yellow/VSC context, prior SC/VSC activity, pace and sector disruption, gaps/close battles, pit activity, tyre state, and weather. `Deleted` laps are intentionally not called retirements. Current SC-active laps are not eligible origins for a *new* SC hazard.

In [ ]:
LIVE=['yellow_active','vsc_active','previous_sc_count','previous_vsc_count','laps_since_sc','laps_since_vsc','active_cars','median_lap_time','lap_time_std','median_sector_1','median_sector_2','median_sector_3','pace_abnormality','median_gap_ahead','close_battle_rate','recent_pits_3','median_tyre_life','rainfall','track_temp','air_temp','weather_change']
def landmark_rows(s, with_target=True):
    rows=[]
    for rid,full_race in s.sort_values(['race_id','lap']).groupby('race_id'):
        starts=full_race.set_index('lap').sc_start.to_dict(); end=int(full_race.lap.max())
        for q in full_race.loc[~full_race.sc_active].itertuples(index=False):
            for k in range(1,MAX_H+1):
                if with_target and q.lap+k>end: break
                d={f:getattr(q,f) for f in LIVE}; d.update(race_id=rid,circuit=q.circuit,lap=q.lap,offset=k,forecast_progress=min(1.0,(q.lap+k)/q.planned_race_laps))
                if with_target:
                    d['y']=int(bool(starts.get(q.lap+k,False))); rows.append(d)
                    if d['y']: break
                else: rows.append(d)
    return pd.DataFrame(rows)
def attach_prior(long):
    table,keys,fallback=circuit_prior; x=long.copy(); x['bin']=pd.cut(x.forecast_progress,np.linspace(0,1,13),include_lowest=True)
    return x.merge(table,on=keys,how='left').rename(columns={'p':'prior'}).assign(prior=lambda z:z.prior.fillna(fallback))

imputer=SimpleImputer(strategy='median'); scaler=StandardScaler()
train_long=attach_prior(landmark_rows(train)); val_long=attach_prior(landmark_rows(val)); X_train=scaler.fit_transform(imputer.fit_transform(train_long[LIVE])); X_val=scaler.transform(imputer.transform(val_long[LIVE]))
def fit_offset_update(x,y,prior,penalty=1.0):
    x=np.c_[np.ones(len(x)),x]; off=logit(np.clip(prior,1e-5,1-1e-5))
    def fun(b):
        z=off+x@b; return np.mean(np.logaddexp(0,z)-y*z)+.5*penalty*np.sum(b[1:]**2)/len(y)
    def jac(b):
        p=expit(off+x@b); g=x.T@(p-y)/len(y); g[1:]+=penalty*b[1:]/len(y); return g
    return minimize(fun,np.zeros(x.shape[1]),jac=jac,method='L-BFGS-B').x
beta=fit_offset_update(X_train,train_long.y.to_numpy(),train_long.prior.to_numpy())
def raw_update(x,prior): return expit(logit(np.clip(prior,1e-5,1-1e-5))+np.c_[np.ones(len(x)),x]@beta)
val_raw=raw_update(X_val,val_long.prior.to_numpy()); z=logit(np.clip(val_raw,1e-5,1-1e-5)); cal_beta=fit_offset_update(z.reshape(-1,1),val_long.y.to_numpy(),np.repeat(.5,len(z)),penalty=1e-6)
def calibrated_update(x,prior): return expit(np.c_[np.ones(len(x)),logit(np.clip(raw_update(x,prior),1e-5,1-1e-5))]@cal_beta)
print('train landmark prevalence:',train_long.y.mean(),'validation landmark prevalence:',val_long.y.mean())

## Sequential replay, multi-horizon probabilities, and output

At each test lap only its state and historical training tables are used. Future rows are used only after prediction to score censored horizons.

In [ ]:
def score_race_states(s):
    long=attach_prior(landmark_rows(s,with_target=False)); X=scaler.transform(imputer.transform(long[LIVE])); long['updated_hazard']=calibrated_update(X,long.prior.to_numpy()); long['historical_hazard']=long.prior
    out=s.loc[~s.sc_active,['race_id','lap','target_1','target_3','target_5','target_10']].copy(); key=out.race_id+'::'+out.lap.astype(str)
    for label in ['historical','updated']:
        wide=long.pivot(index=['race_id','lap'],columns='offset',values=f'{label}_hazard').reindex(columns=range(1,MAX_H+1)); arr=wide.to_numpy(); prob={h:1-np.prod(1-arr[:,:h],axis=1) for h in HORIZONS}; aligned=wide.index.get_level_values(0).astype(str)+'::'+wide.index.get_level_values(1).astype(str); pos=pd.Series(np.arange(len(aligned)),index=aligned); ix=key.map(pos)
        for h,v in prob.items(): out[f'{label}_P_SC_{h}']=np.where(ix.notna(),v[ix.fillna(0).astype(int)],np.nan)
    out['historical_SC_probability']=out.historical_P_SC_1; out['updated_SC_probability']=out.updated_P_SC_1
    return out
predictions=score_race_states(test)
assert (predictions.updated_P_SC_1<=predictions.updated_P_SC_3).all() and (predictions.updated_P_SC_3<=predictions.updated_P_SC_5).all() and (predictions.updated_P_SC_5<=predictions.updated_P_SC_10).all()
predictions[['race_id','lap','historical_SC_probability','updated_SC_probability','updated_P_SC_1','updated_P_SC_3','updated_P_SC_5','updated_P_SC_10']].head()

## Evaluation, calibration, and risk wave

A model is not considered calibrated merely because a calibration step was applied. The held-out metrics and reliability curve below are the evidence.

In [ ]:
def metrics(y,p):
    x=pd.DataFrame({'y':y,'p':p}).dropna(); y=x.y.astype(int); p=x.p.clip(1e-6,1-1e-6); return dict(Brier=brier_score_loss(y,p),LogLoss=log_loss(y,p),PR_AUC=average_precision_score(y,p),ROC_AUC=roc_auc_score(y,p))
rows=[]
for h in HORIZONS:
    for name in ['historical','updated']:
        rows.append({'horizon':h,'model':name,**metrics(predictions[f'target_{h}'],predictions[f'{name}_P_SC_{h}'])})
evaluation=pd.DataFrame(rows).set_index(['horizon','model']); display(evaluation.round(5))
cal=predictions.dropna(subset=['target_1','updated_P_SC_1']); obs,est=calibration_curve(cal.target_1,cal.updated_P_SC_1,n_bins=8,strategy='quantile'); fig,ax=plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1],'--',c='grey'); ax.plot(est,obs,'o-'); ax.set(title='Held-out calibration: updated next-lap risk',xlabel='Predicted probability',ylabel='Observed frequency'); plt.show()
example=test.groupby('race_id').sc_start.sum(); example_id=example[example.gt(0)].index[0]; wave=predictions[predictions.race_id.eq(example_id)].merge(test[['race_id','lap','race_name','sc_start']],on=['race_id','lap']); fig,ax=plt.subplots(figsize=(11,4)); ax.plot(wave.lap,wave.historical_P_SC_5,label='historical P_SC_5'); ax.plot(wave.lap,wave.updated_P_SC_5,label='updated P_SC_5'); [ax.axvline(x,c='tab:red',ls='--') for x in wave.loc[wave.sc_start,'lap']]; ax.set(title=f'Risk wave — {wave.race_name.iloc[0]}',xlabel='Completed lap',ylabel='P(new SC within 5 laps)'); ax.legend(); plt.show()

In [ ]:
# Existing XGBoost benchmark. It is evaluated only when the saved artifacts are present.
# Its feature contract differs from the temporary adapter, so this is a benchmark, not a replacement.
try:
    import joblib
    model_path=Path('models/sc_trigger_xgboost.joblib'); calibrator_path=Path('models/sc_trigger_calibrator.joblib'); features_path=Path('models/sc_trigger_features.joblib')
    if not all(p.exists() for p in [model_path,calibrator_path,features_path]): raise FileNotFoundError('Saved XGBoost artifacts unavailable')
    source=test.loc[~test.sc_active].copy(); names=joblib.load(features_path)
    xgb_input=pd.DataFrame({'LapNumber':source.lap,'race_progress':source.race_progress,'tyre_age_med':source.median_tyre_life,'lap_time_std':source.lap_time_std,'retirements':0,'total_pits':source.groupby('race_id').pit_stops_lap.cumsum(),'pits_last3':source.recent_pits_3,'track_temp':source.track_temp,'air_temp':source.air_temp,'humidity':0,'rainfall_flag':source.rainfall.fillna(0).gt(0).astype(int),'sc_count_prev':source.previous_sc_count,'laps_since_last_sc':source.laps_since_sc,'n_cars':source.active_cars})[names].fillna(0)
    raw=joblib.load(model_path).predict_proba(xgb_input)[:,1]; xgb_p=np.asarray(joblib.load(calibrator_path).predict(raw)).reshape(-1)
    xgb_eval=metrics(source.target_1,xgb_p); display(pd.DataFrame([xgb_eval],index=['Existing XGBoost benchmark']).round(5))
except (ImportError,FileNotFoundError) as exc: print('XGBoost benchmark skipped:',exc)


## Inspect risk estimation for one race

Set `RACE_TO_INSPECT` to a held-out `race_id` below (for example `2025_01`). Run the cell to see the sequential probability estimates for that race. The vertical red lines are actual SC starts and are shown only afterward for evaluation context.

To keep this honest, the selector uses the held-out test races already scored by the chronological replay.

In [ ]:
# Set a held-out race_id from AVAILABLE_RACES, or leave None to choose a test race containing an SC.
AVAILABLE_RACES = test[['race_id','year','round','race_name']].drop_duplicates().sort_values(['year','round'])
display(AVAILABLE_RACES)
RACE_TO_INSPECT = None  # Example: '2025_01'

def show_risk_estimation(race_id=RACE_TO_INSPECT):
    event_counts = test.groupby('race_id').sc_start.sum()
    if race_id is None:
        race_id = event_counts[event_counts.gt(0)].index[0]
    if race_id not in set(AVAILABLE_RACES.race_id):
        raise ValueError(f'Unknown held-out race_id: {race_id}. Choose one from AVAILABLE_RACES.')
    state = test[test.race_id.eq(race_id)][['race_id','lap','race_name','sc_start','yellow_active','vsc_active','close_battle_rate','pace_abnormality']].copy()
    view = predictions[predictions.race_id.eq(race_id)].merge(state, on=['race_id','lap'], how='left')
    for horizon in HORIZONS:
        view[f'P_SC_{horizon}'] = view[f'updated_P_SC_{horizon}']
    title = view.race_name.iloc[0]
    print(f'Risk estimation: {title} ({race_id})')
    display(view[['race_id','lap','historical_SC_probability','updated_SC_probability','P_SC_1','P_SC_3','P_SC_5','P_SC_10','sc_start']].round(4))
    fig,(ax1,ax2)=plt.subplots(2,1,figsize=(12,7),sharex=True,gridspec_kw={'height_ratios':[3,1]})
    ax1.plot(view.lap,view.historical_P_SC_5,label='Historical P(SC within 5)',color='grey',alpha=.8)
    for horizon in HORIZONS: ax1.plot(view.lap,view[f'P_SC_{horizon}'],label=f'Updated P(SC within {horizon})')
    for lap in view.loc[view.sc_start,'lap']: ax1.axvline(lap,color='tab:red',ls='--',alpha=.75,label='Actual SC start')
    handles,labels=ax1.get_legend_handles_labels(); unique=dict(zip(labels,handles)); ax1.legend(unique.values(),unique.keys(),ncol=2)
    ax1.set(title=f'Sequential SC risk estimation — {title}',ylabel='Probability',ylim=(0,None))
    ax2.step(view.lap,view.yellow_active.astype(int),where='mid',label='Yellow active',color='tab:orange')
    ax2.step(view.lap,view.vsc_active.astype(int),where='mid',label='VSC active',color='tab:purple')
    ax2.set(xlabel='Completed lap',ylabel='Observed context',yticks=[0,1]); ax2.legend(loc='upper right'); plt.tight_layout(); plt.show()
    return view

race_risk = show_risk_estimation()
# Plain output aliases required by the Monte Carlo interface.
for horizon in HORIZONS: predictions[f'P_SC_{horizon}'] = predictions[f'updated_P_SC_{horizon}']


## Abu Dhabi 2021 case-study replay

This replay is included to inspect the model's risk estimates around the known VSC (lap 34) and SC start (lap 51). Abu Dhabi 2021 belongs to the current training period, so this is a **logic and visualisation case study only**, not a held-out performance result.

In [ ]:
ABU_DHABI_2021_RACE_ID = '2021_22'
abu_state = states[states.race_id.eq(ABU_DHABI_2021_RACE_ID)].copy()
assert abu_state.race_name.iloc[0] == 'Abu Dhabi Grand Prix'
abu_dhabi_2021_risk = score_race_states(abu_state).merge(abu_state[['race_id','lap','race_name','sc_start','vsc_start','yellow_active','vsc_active']],on=['race_id','lap'],how='left')
for horizon in HORIZONS: abu_dhabi_2021_risk[f'P_SC_{horizon}'] = abu_dhabi_2021_risk[f'updated_P_SC_{horizon}']
display(abu_dhabi_2021_risk[['race_id','lap','historical_SC_probability','updated_SC_probability','P_SC_1','P_SC_3','P_SC_5','P_SC_10','vsc_start','sc_start']].round(4))
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(12,7),sharex=True,gridspec_kw={'height_ratios':[3,1]})
ax1.plot(abu_dhabi_2021_risk.lap,abu_dhabi_2021_risk.historical_P_SC_5,label='Historical P(SC within 5)',color='grey')
for horizon in HORIZONS: ax1.plot(abu_dhabi_2021_risk.lap,abu_dhabi_2021_risk[f'P_SC_{horizon}'],label=f'Updated P(SC within {horizon})')
for lap in abu_dhabi_2021_risk.loc[abu_dhabi_2021_risk.vsc_start,'lap']: ax1.axvline(lap,color='tab:purple',ls=':',label='Actual VSC start')
for lap in abu_dhabi_2021_risk.loc[abu_dhabi_2021_risk.sc_start,'lap']: ax1.axvline(lap,color='tab:red',ls='--',label='Actual SC start')
handles,labels=ax1.get_legend_handles_labels(); unique=dict(zip(labels,handles)); ax1.legend(unique.values(),unique.keys(),ncol=2); ax1.set(title='Abu Dhabi GP 2021 — SC risk case study',ylabel='Probability',ylim=(0,None))
ax2.step(abu_dhabi_2021_risk.lap,abu_dhabi_2021_risk.yellow_active.astype(int),where='mid',label='Yellow active',color='tab:orange'); ax2.step(abu_dhabi_2021_risk.lap,abu_dhabi_2021_risk.vsc_active.astype(int),where='mid',label='VSC active',color='tab:purple'); ax2.set(xlabel='Completed lap',ylabel='Observed context',yticks=[0,1]); ax2.legend(); plt.tight_layout(); plt.show()


## Monte Carlo and Race State Manager handoff

`predictions` supplies `race_id`, `lap`, `historical_SC_probability`, `updated_SC_probability`, `P_SC_1`, `P_SC_3`, `P_SC_5`, and `P_SC_10`. The simulator should use these as SC scenario weights, not as a strategy decision or an SC-duration prediction.

The historical prior remains the production fallback unless the updated model has a reliable held-out improvement in Brier score and log loss. The temporary adapter must eventually be replaced with RaceState_t fields, especially official scheduled laps, authoritative retirements, incident/yellow context, and live weather/field data.